<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.6-rag-with-llamaindex/notebooks/exercises-6_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.6: RAG with LlamaIndex

*Module 6 · Retrieval-Augmented Generation (RAG)*

> LangChain is the general-purpose LLM framework. LlamaIndex is the data-specialist — purpose-built for connecting LLMs to your data. Its abstractions ( VectorStoreIndex , QueryEngine ) make RAG even more concise. This lesson rebuilds the same pipeline with LlamaIndex, then compares both frameworks head-to-head.

`LlamaIndex` · `Gemini` · `VectorStoreIndex` · `vs LangChain` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — LlamaIndex RAG in 10 Lines — Even Simpler — `01_llamaindex_rag.py`
2. Step 2 — Customizing the Pipeline — Chunk Size, Prompts, Retrieval — `02_custom_pipeline.py`
3. Step 3 — Chat Engine — Conversational RAG with Memory — `03_chat_engine.py`
4. Step 4 — Head-to-Head — LangChain vs LlamaIndex vs Scratch — `04_side_by_side.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q llama-index


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · LlamaIndex RAG in 10 Lines — Even Simpler

VectorStoreIndex handles chunking, embedding, and retrieval in one abstraction.

**`01_llamaindex_rag.py`** · _Block 1 of 4_

LLAMAINDEX RAG — Full pipeline in ~10 lines — pip install llama-index llama-index-llms-gemini llama-index-embeddings-gemini


In [ ]:
# LLAMAINDEX RAG — Full pipeline in ~10 lines
# pip install llama-index llama-index-llms-gemini llama-index-embeddings-gemini
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

# Configure LLM and embedding model globally
Settings.llm = Gemini(model="models/gemini-2.0-flash")
Settings.embed_model = GeminiEmbedding(model_name="models/text-embedding-004")

# 1. Create documents
documents = [
    Document(text="Netsetos refund policy: full refund within 7 days. 50% from 8-30 days. No refunds after 30 days."),
    Document(text="GenAI course: 14999 rupees. Lifetime access, Discord, weekly live sessions, certificate, 5000 GCP credits."),
    Document(text="Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. Python + GCP. 85% completion rate."),
    Document(text="Prerequisites: basic Python and high school math. No ML experience needed. 146 hours, 14 modules."),
]

# 2. Build index (auto: chunk + embed + store)
index = VectorStoreIndex.from_documents(documents)

# 3. Create query engine
engine = index.as_query_engine(similarity_top_k=3)

# 4. Query!
print("LlamaIndex RAG:\n")
for q in ["What is the refund policy?", "How much does the course cost?", "What are the prerequisites?"]:
    resp = engine.query(q)
    print(f"  Q: {q}")
    print(f"  A: {str(resp)[:120]}")
    print(f"  Sources: {[n.metadata for n in resp.source_nodes]}\n")


> **What just happened?** VectorStoreIndex.from_documents() did THREE things in one call: chunked the documents, embedded each chunk, and stored them in an in-memory vector store. as_query_engine() created a retriever + prompt + LLM pipeline. 10 lines. No manual chunking, no manual embedding, no chain composition. LlamaIndex’s opinionated defaults handle everything.


### Step 2 · Customizing the Pipeline — Chunk Size, Prompts, Retrieval

LlamaIndex is simple by default, but fully customizable when you need control.

**`02_custom_pipeline.py`** · _Block 2 of 4_

LLAMAINDEX — Customized pipeline


In [ ]:
# LLAMAINDEX — Customized pipeline
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

Settings.llm = Gemini(model="models/gemini-2.0-flash")
Settings.embed_model = GeminiEmbedding(model_name="models/text-embedding-004")

# ── Custom chunking ──
splitter = SentenceSplitter(chunk_size=128, chunk_overlap=20)

docs = [
    Document(text="Netsetos is an edtech company in Hyderabad. We offer GenAI, Data Science, and Cloud courses. Our flagship GenAI course has 14 modules spanning 146 hours.", metadata={"source": "about.txt"}),
    Document(text="Refund policy: full refund within 7 days. 50% from 8-30 days. None after 30 days. Contact [email protected].", metadata={"source": "refund.txt"}),
    Document(text="GenAI course costs 14999 rupees. Includes lifetime access, Discord community, weekly live doubt sessions, and 5000 GCP credits.", metadata={"source": "pricing.txt"}),
]

# ── Build index with custom splitter ──
index = VectorStoreIndex.from_documents(docs, transformations=[splitter])

# ── Custom prompt template ──
custom_prompt = PromptTemplate(
    "You are a Netsetos support assistant. Answer ONLY from context.\n"
    "If not found, say 'I don't have this information.'\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

engine = index.as_query_engine(
    similarity_top_k=2,
    text_qa_template=custom_prompt,
)

print("Customized LlamaIndex RAG:\n")
for q in ["What courses does Netsetos offer?", "What is the refund policy?", "Do you offer blockchain courses?"]:
    resp = engine.query(q)
    sources = [n.metadata.get("source","?") for n in resp.source_nodes]
    print(f"  Q: {q}")
    print(f"  A: {str(resp)[:120]}")
    print(f"  Sources: {sources}\n")


### Step 3 · Chat Engine — Conversational RAG with Memory

LlamaIndex has a built-in chat engine that maintains conversation history.

**`03_chat_engine.py`** · _Block 3 of 4_

LLAMAINDEX CHAT ENGINE — Conversational RAG


In [ ]:
# LLAMAINDEX CHAT ENGINE — Conversational RAG
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")
Settings.llm = Gemini(model="models/gemini-2.0-flash")
Settings.embed_model = GeminiEmbedding(model_name="models/text-embedding-004")

docs = [
    Document(text="Refund policy: full refund within 7 days. 50% from 8-30 days. None after 30 days."),
    Document(text="GenAI course: 14999 rupees. 146 hours, 14 modules. Python + GCP."),
    Document(text="Prerequisites: basic Python and high school math. No ML experience needed."),
]
index = VectorStoreIndex.from_documents(docs)

# as_chat_engine: auto conversation memory + RAG retrieval
chat = index.as_chat_engine(chat_mode="condense_plus_context")

print("LlamaIndex Chat Engine:\n")
conversation = [
    "What are the prerequisites?",
    "And how much does it cost?",       # "it" refers to the course
    "Can I get a refund after 2 weeks?",  # follow-up
]
for q in conversation:
    resp = chat.chat(q)
    print(f"  You: {q}")
    print(f"  Bot: {str(resp)[:120]}\n")

print("  'condense_plus_context' mode:")
print("  1. Condenses follow-up + history into standalone query")
print("  2. Retrieves relevant context")
print("  3. Generates answer with full conversation awareness")


> **What just happened?** as_chat_engine(chat_mode="condense_plus_context") gave us conversational RAG in ONE line. When the user asks “how much does IT cost?”, the engine condenses the history (“it” = the GenAI course from the previous question), retrieves the pricing document, and answers. LangChain needed MessagesPlaceholder + manual history. LlamaIndex: one parameter.


### Step 4 · Head-to-Head — LangChain vs LlamaIndex vs Scratch

Same RAG pipeline, three implementations. Code lines, features, and when to use each.

**`04_side_by_side.py`** · _Block 4 of 4_

SIDE-BY-SIDE: Same RAG, 3 approaches


In [ ]:
# SIDE-BY-SIDE: Same RAG, 3 approaches

print("Same RAG pipeline, 3 implementations:\n")

code_comparison = {
    "From Scratch": """
    rag = SimpleRAG()
    rag.add_document(text)
    result = rag.query("What is the refund policy?")
    """,
    "LangChain": """
    vectorstore = Chroma.from_documents(docs, embeddings)
    chain = {"context": retriever | format, "question": passthrough} | prompt | llm | parser
    answer = chain.invoke("What is the refund policy?")
    """,
    "LlamaIndex": """
    index = VectorStoreIndex.from_documents(docs)
    engine = index.as_query_engine()
    response = engine.query("What is the refund policy?")
    """,
}

for name, code in code_comparison.items():
    lines = [l for l in code.strip().split("\n") if l.strip()]
    print(f"  {name} ({len(lines)} core lines):")
    for l in lines: print(f"    {l.strip()}")
    print()

print("All three produce the same answer.")
print("Choose based on: control needed, team skills, primary use case.")


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
